# BLIP-2 query-count ablation: CUDA / ROCm

Evaluate the original pretrained `Salesforce/blip2-flan-t5-xl` model without training on VQAv2. Run from the project root or `scripts/`, using the project's BF16-capable NVIDIA CUDA or AMD ROCm Jupyter kernel. This notebook calls `load_blip2.ipynb` in the current kernel and reuses the cached model; the loader's 12 GiB placement budget is unchanged. Shut down any other kernel holding a GPU copy of the model.

| K | Condition | Runs |
|---:|---|---:|
| 8 | Random nested subsets of pretrained queries | 5 |
| 16 | Random nested subsets of pretrained queries | 5 |
| 32 | Unmodified pretrained baseline | 1 |
| 64 | Repeat all 32 input query embeddings twice | 1 |
| 128 | Repeat all 32 input query embeddings four times | 1 |

For a given seed, the K=8 selection is contained in the K=16 selection. K=64/128 are repeated-query controls **before the Q-Former**, not independently trained higher-capacity models. All 13 conditions use the same questions and five-beam decoding. The single-question greedy `baseline_test.ipynb` is a separate functionality check.

Start with `smoke`, then optionally run `pilot` and finally `main`. The pilot is a prefix of the existing sorted seed-42 subset, not an independent evaluation sample. Each stage and hardware/software configuration gets a separate signed results directory. Predictions are saved after every question; rerun with identical settings and saved notebook code to resume.

After a completed run, use `score_results.ipynb` with the same stage and the exact results directory printed below. Leave-one-query-out (K=31) is a separate future notebook, not part of this 13-condition run.

## 1. Configuration and utilities

Save the notebook before running so the manifest hashes the intended code. Set `STAGE` in the next cell (or `BLIP2_STAGE` before starting Jupyter). The default is deliberately `smoke`.

In [ ]:
import os
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")  # Preserve an existing user override.

from pathlib import Path
import hashlib
import json
import platform
import random
import time
import uuid
import math
import weakref
from datetime import datetime, timezone
from importlib.metadata import version

import nbformat
import torch
from filelock import FileLock
from PIL import Image
from tqdm.auto import tqdm

STAGE = os.environ.get("BLIP2_STAGE", "smoke")  # Change default to pilot/main when ready.
QUESTION_COUNTS = {"smoke": 10, "pilot": 500, "main": 2000}
QUERY_SEEDS = [0, 1, 2, 3, 4]
RUN_ORDER_SEED = 2026
WARMUP_QUESTIONS = 2
PIPELINE_VERSION = 2
PROMPT = "Question: {question} Short answer:"
GENERATION_SETTINGS = {
    "do_sample": False,
    "num_beams": 5,
    "length_penalty": -1.0,
    "max_new_tokens": 10,
}

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA-enabled or ROCm-enabled PyTorch environment is required.")
GPU_BACKEND = "rocm" if torch.version.hip else "cuda" if torch.version.cuda else None
if GPU_BACKEND is None:
    raise RuntimeError("Could not identify a CUDA or ROCm PyTorch build.")

if STAGE not in QUESTION_COUNTS:
    raise ValueError("STAGE must be smoke, pilot, or main.")
if len(QUERY_SEEDS) != len(set(QUERY_SEEDS)) or not QUERY_SEEDS:
    raise ValueError("Query seeds must be unique and nonempty.")

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "scripts" else cwd
DATA_DIR = PROJECT_ROOT / "data" / "validation_dataset"
IMAGE_DIR = DATA_DIR / "images" / "val2014"
RESULTS_ROOT = PROJECT_ROOT / "results" / "query_ablation"


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json_atomic(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".partial")
    try:
        with temporary.open("w", encoding="utf-8") as stream:
            json.dump(value, stream, indent=2, allow_nan=False)
            stream.write("\n")
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def image_path(question):
    return IMAGE_DIR / f"COCO_val2014_{int(question['image_id']):012d}.jpg"


## 2. Load and validate the fixed evaluation subset

In [ ]:
questions_path = DATA_DIR / "v2_OpenEnded_mscoco_val2014_questions_subset.json"
annotations_path = DATA_DIR / "v2_mscoco_val2014_annotations_subset.json"
all_questions = json.loads(questions_path.read_text(encoding="utf-8"))["questions"]
annotations = json.loads(annotations_path.read_text(encoding="utf-8"))["annotations"]
annotation_by_id = {int(item["question_id"]): item for item in annotations}
if len(annotation_by_id) != len(annotations):
    raise ValueError("Duplicate annotation question IDs.")
if len(all_questions) != 2000:
    raise ValueError("Expected the existing 2,000-question validation subset.")
questions = all_questions[:QUESTION_COUNTS[STAGE]]
question_ids = [int(question["question_id"]) for question in questions]
if len(set(question_ids)) != len(question_ids):
    raise ValueError("Duplicate question IDs in the selected data.")
for question in questions:
    annotation = annotation_by_id[int(question["question_id"])]
    if int(annotation["image_id"]) != int(question["image_id"]):
        raise ValueError("Question and annotation image IDs differ.")
    if len(annotation["answers"]) != 10:
        raise ValueError("Official VQA scoring requires all ten annotator answers.")
    if not image_path(question).is_file():
        raise FileNotFoundError(image_path(question))

image_hashes = {path.name: file_sha256(path) for path in sorted({image_path(q) for q in questions})}
print(f"Stage {STAGE}: {len(questions)} questions, {len(image_hashes)} distinct images.")


## 3. Reuse the pretrained model and define query conditions

PyTorch exposes both NVIDIA CUDA and AMD ROCm GPUs through `torch.cuda`. The notebook detects the backend automatically; no GPU-specific device index is hard-coded in the ablation. The loader is still responsible for placing the model in memory. No weights are trained.

In [ ]:
# Restore only a query parameter that belongs to this exact model instance.
# A fresh/replaced model must never receive a parameter from a previous model.
if "_ablation_model_ref" in globals() and _ablation_model_ref() is globals().get("model") and globals().get("model") is not None:
    restore_original_queries()

loader = PROJECT_ROOT / "scripts" / "load_blip2.ipynb"
get_ipython().run_line_magic("run", f'-i "{loader.as_posix()}"')

if tuple(model.query_tokens.shape) != (1, 32, 768) or model.config.num_query_tokens != 32 or processor.num_query_tokens != 32:
    raise RuntimeError("Start from the original 32-query checkpoint in a fresh kernel.")
if processor.tokenizer.convert_tokens_to_ids("<image>") != model.config.image_token_index:
    raise RuntimeError("The processor and model image-token IDs must match.")

vision_parameter = next(model.vision_model.parameters())
DEVICE = vision_parameter.device
DTYPE = vision_parameter.dtype
if DEVICE.type != "cuda":
    raise RuntimeError("The vision encoder must be on the GPU.")
with torch.cuda.device(DEVICE):
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError("The selected GPU must support BF16 inference.")
if DTYPE != torch.bfloat16:
    raise RuntimeError(f"Expected BF16 vision inputs, got {DTYPE}.")
# A single resident GPU is required for comparable timing/memory metrics and
# safe query replacement. Do not silently benchmark CPU/disk offloading.
parameter_devices = {parameter.device for parameter in model.parameters()}
if parameter_devices != {DEVICE}:
    raise RuntimeError(
        f"Expected the whole model on {DEVICE}; found {parameter_devices}. "
        "Stop other GPU kernels and check the shared loader's placement before proceeding."
    )

model.requires_grad_(False)
model.eval()
_ablation_model_ref = weakref.ref(model)
ablation_original_parameter = model.query_tokens
original_queries = ablation_original_parameter.detach().clone()


def query_indices(k, seed=None):
    """Return zero-based query IDs; paired seeds produce nested subsets."""
    if k not in (8, 16, 32, 64, 128):
        raise ValueError("Unsupported query count.")
    if k < 32:
        if seed is None:
            raise ValueError("Pruning requires a query-selection seed.")
        generator = torch.Generator(device="cpu").manual_seed(seed)
        return torch.randperm(32, generator=generator, device="cpu")[:k].sort().values
    return torch.arange(32, device="cpu").repeat(k // 32)


def restore_original_queries():
    """Restore the original parameter, configuration and processor; verify values."""
    if _ablation_model_ref() is not model:
        raise RuntimeError("Model instance changed; restart from the pretrained checkpoint.")
    model.query_tokens = ablation_original_parameter
    model.config.num_query_tokens = 32
    processor.num_query_tokens = 32
    if (tuple(model.query_tokens.shape) != (1, 32, 768)
            or model.config.num_query_tokens != 32
            or processor.num_query_tokens != 32
            or not torch.equal(model.query_tokens.detach(), original_queries)):
        raise RuntimeError("Original query restoration failed. Restart the kernel.")


def set_query_count(k, seed=None):
    """Always select from the untouched pretrained queries, never a prior condition."""
    if _ablation_model_ref() is not model:
        raise RuntimeError("Model instance changed; rerun the model setup cell.")
    indices = query_indices(k, seed)
    if k == 32:
        restore_original_queries()
    else:
        selected = original_queries.index_select(1, indices.to(original_queries.device))
        model.query_tokens = torch.nn.Parameter(selected, requires_grad=False)
        model.config.num_query_tokens = k
        processor.num_query_tokens = k
    if tuple(model.query_tokens.shape) != (1, k, 768) or model.config.num_query_tokens != k or processor.num_query_tokens != k:
        raise RuntimeError("Query count and processor configuration are inconsistent.")
    return indices.tolist()


def condition_name(k, seed):
    return f"k{k:03d}-seed{seed:03d}" if seed is not None else f"k{k:03d}"


conditions = [(k, seed) for k in (8, 16) for seed in QUERY_SEEDS]
conditions += [(32, None), (64, None), (128, None)]
random.Random(RUN_ORDER_SEED).shuffle(conditions)

# Validate every query selection without running generation or changing model state.
for seed in QUERY_SEEDS:
    assert set(query_indices(8, seed).tolist()) <= set(query_indices(16, seed).tolist())
for k, seed in conditions:
    assert len(query_indices(k, seed)) == k

print(f"{GPU_BACKEND.upper()}: {torch.cuda.get_device_name(DEVICE)} | {DTYPE} | {len(conditions)} conditions")

## Measurement protocol

Batch size is one. Each unfinished condition receives two unmeasured warmup questions (also on resume). Generation time covers `model.generate()` with GPU synchronization before and after. Image reading, preprocessing, transfers, decoding and result writing are excluded. Throughput is measured questions divided by summed generation time, not total wall-clock runtime.

Peak allocated and reserved GPU memory are reset for each question after input transfer; allocated memory includes the resident model and inputs. The difference between peak allocated and pre-generation allocated memory can be computed from the records. Generated length excludes the Flan-T5 decoder start token and padding, and includes EOS when emitted.

Fixed seeds govern query selection and condition order, not a guarantee of bitwise-identical outputs across CUDA/ROCm hardware. The 12 GiB model-placement budget is not a hard limit on runtime allocations. The complete model must reside on one GPU; do not compare offloaded or multi-GPU runs using these metrics.

Keep latency and memory comparisons within the same GPU and software configuration. An interrupted run may combine timings from different sessions on the same hardware, so use a fresh uninterrupted run if precise performance benchmarking is the main goal. Accuracy comparisons require identical question IDs, prompt, generation settings, checkpoint and query indices.

## 4. Record the experiment manifest

Hardware, model revision, source code, dataset and condition settings contribute to the signature. Existing AMD results remain untouched. Changes to these fields intentionally start a new experiment directory.

In [ ]:
checkpoint_revision = getattr(model.config, "_commit_hash", None)
if not checkpoint_revision:
    raise RuntimeError("Could not identify the loaded Hugging Face checkpoint revision.")
notebook = nbformat.read(PROJECT_ROOT / "scripts" / "query_ablation.ipynb", as_version=4)
code_digest = hashlib.sha256("\n".join(cell.source for cell in notebook.cells if cell.cell_type == "code").encode()).hexdigest()
packages = ["torch", "torchvision", "transformers", "accelerate", "huggingface_hub", "numpy", "Pillow", "nbformat", "filelock", "tqdm"]
experiment = {
    "pipeline_version": PIPELINE_VERSION,
    "notebook_code_sha256": code_digest,
    "loader_sha256": file_sha256(loader),
    "stage": STAGE,
    "model_id": MODEL_ID,
    "checkpoint_revision": checkpoint_revision,
    "original_query_sha256": hashlib.sha256(original_queries.cpu().contiguous().view(torch.uint8).numpy().tobytes()).hexdigest(),
    "question_file_sha256": file_sha256(questions_path),
    "annotation_file_sha256": file_sha256(annotations_path),
    "question_ids": question_ids,
    "image_sha256": image_hashes,
    "prompt": PROMPT,
    "generation": GENERATION_SETTINGS,
    "batch_size": 1,
    "warmup_questions": WARMUP_QUESTIONS,
    "precision": str(DTYPE),
    "device_map": {key: str(value) for key, value in model.hf_device_map.items()},
    "gpu_name": torch.cuda.get_device_name(DEVICE),
    "gpu_memory_bytes": torch.cuda.get_device_properties(DEVICE).total_memory,
    "gpu_backend": GPU_BACKEND,
    "cuda_version": torch.version.cuda,
    "rocm_version": torch.version.hip,
    "gpu_device_index": DEVICE.index,
    "query_dtype": str(original_queries.dtype),
    "parameter_dtypes": sorted({str(p.dtype) for p in model.parameters()}),
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "packages": {name: version(name) for name in packages},
    "run_order_seed": RUN_ORDER_SEED,
    "query_seeds": QUERY_SEEDS,
    "conditions": [
        {"name": condition_name(k, seed), "k": k, "seed": seed,
         "query_indices": query_indices(k, seed).tolist(),
         "kind": "pruning" if k < 32 else "baseline" if k == 32 else "repeated-query control"}
        for k, seed in conditions
    ],
}
signature = hashlib.sha256(json.dumps(experiment, sort_keys=True).encode()).hexdigest()
EXPERIMENT_DIR = RESULTS_ROOT / f"{STAGE}-{signature[:16]}"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
print("Results directory:", EXPERIMENT_DIR)
print(f"Planned predictions: {len(conditions) * len(questions):,}")


## 5. Run or resume the experiment

A file lock prevents concurrent writers. Every completed prediction is flushed to disk; an incomplete final JSONL line is recovered on rerun. Only completed conditions are exported to the official VQA prediction format. The original query parameter is restored after each condition and on exit.

In [ ]:
def read_records(path, expected_ids, condition):
    """Recover an interrupted final write and validate the saved ordered prefix."""
    if not path.exists():
        return []
    records = []
    with path.open("rb+") as stream:
        while True:
            start = stream.tell()
            line = stream.readline()
            if not line:
                break
            if not line.endswith(b"\n"):
                stream.truncate(start)
                stream.flush()
                os.fsync(stream.fileno())
                break
            record = json.loads(line)
            position = len(records)
            if position >= len(expected_ids) or record["question_id"] != expected_ids[position]:
                raise ValueError(f"Unexpected or duplicate question in {path}.")
            if record["k"] != condition["k"] or record["query_seed"] != condition["seed"]:
                raise ValueError(f"Saved condition differs in {path}.")
            if record["image_id"] != int(annotation_by_id[record["question_id"]]["image_id"]):
                raise ValueError(f"Saved image ID differs in {path}.")
            if (not isinstance(record["answer"], str)
                    or not math.isfinite(record["generation_seconds"])
                    or record["generation_seconds"] <= 0):
                raise ValueError(f"Invalid prediction record in {path}.")
            records.append(record)
    return records


def generate_answer(question):
    with Image.open(image_path(question)) as image:
        inputs = processor(
            images=image.convert("RGB"),
            text=PROMPT.format(question=question["question"]),
            return_tensors="pt",
        )
    placeholder_count = int((inputs["input_ids"] == model.config.image_token_index).sum())
    if placeholder_count != model.query_tokens.shape[1]:
        raise RuntimeError("Image placeholder count differs from the current query count.")
    inputs = inputs.to(device=DEVICE, dtype=DTYPE)
    torch.cuda.synchronize(DEVICE)
    torch.cuda.reset_peak_memory_stats(DEVICE)
    allocated_before = torch.cuda.memory_allocated(DEVICE)
    started = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(**inputs, **GENERATION_SETTINGS)
    torch.cuda.synchronize(DEVICE)
    elapsed = time.perf_counter() - started
    peak_allocated = torch.cuda.max_memory_allocated(DEVICE)
    peak_reserved = torch.cuda.max_memory_reserved(DEVICE)
    tokens = generated[0, 1:]  # Flan-T5 output begins with a decoder start token.
    length = int((tokens != processor.tokenizer.pad_token_id).sum())
    answer = processor.batch_decode(generated, skip_special_tokens=True)[0].strip()
    return {
        "question_id": int(question["question_id"]),
        "image_id": int(question["image_id"]),
        "answer": answer,
        "generation_seconds": elapsed,
        "generated_token_count": length,
        "allocated_before_bytes": allocated_before,
        "peak_allocated_bytes": peak_allocated,
        "peak_reserved_bytes": peak_reserved,
    }


def run_condition(condition):
    directory = EXPERIMENT_DIR / condition["name"]
    directory.mkdir(exist_ok=True)
    path = directory / "records.jsonl"
    records = read_records(path, question_ids, condition)
    remaining = questions[len(records):]
    try:
        if remaining:
            indices = set_query_count(condition["k"], condition["seed"])
            if indices != condition["query_indices"]:
                raise RuntimeError("Active query indices differ from the manifest.")
            torch.cuda.empty_cache()
            for question in questions[:WARMUP_QUESTIONS]:
                generate_answer(question)
            write_json_atomic(directory / "status.json", {"state": "running", "completed": len(records)})
            with path.open("a", encoding="utf-8") as stream:
                for question in tqdm(remaining, desc=condition["name"]):
                    record = generate_answer(question)
                    record.update(k=condition["k"], query_seed=condition["seed"])
                    stream.write(json.dumps(record, allow_nan=False) + "\n")
                    stream.flush()
                    os.fsync(stream.fileno())
                    records.append(record)
        write_json_atomic(directory / "predictions.json", [
            {"question_id": record["question_id"], "answer": record["answer"]} for record in records
        ])
        write_json_atomic(directory / "status.json", {"state": "complete", "completed": len(records)})
        print(condition["name"], "complete:", len(records), "questions")
    except BaseException as error:
        write_json_atomic(directory / "status.json", {
            "state": "interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
            "completed": len(records), "error": type(error).__name__ + ": " + str(error),
        })
        raise
    finally:
        restore_original_queries()


# A single writer owns this experiment directory, including recovery and exports.
with FileLock(str(EXPERIMENT_DIR / "experiment.lock"), timeout=0):
    manifest_path = EXPERIMENT_DIR / "manifest.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest["signature"] != signature or manifest["experiment"] != experiment:
            raise ValueError("The saved manifest differs from this experiment.")
    else:
        write_json_atomic(manifest_path, {
            "signature": signature, "created_utc": datetime.now(timezone.utc).isoformat(),
            "experiment": experiment,
        })
    # Do not advertise completion until all saved records have been validated.
    (EXPERIMENT_DIR / "complete.json").unlink(missing_ok=True)
    try:
        for condition in experiment["conditions"]:
            run_condition(condition)
        write_json_atomic(EXPERIMENT_DIR / "complete.json", {
            "signature": signature, "condition_count": len(conditions),
            "questions_per_condition": len(questions),
            "finished_utc": datetime.now(timezone.utc).isoformat(),
        })
    finally:
        restore_original_queries()
        torch.cuda.empty_cache()

print(f"Finished {STAGE}: {len(conditions)} conditions. Original K=32 restored.")
print("Score this run using EXPERIMENT_DIR:", EXPERIMENT_DIR)


## 6. Next steps

- For the smoke stage, expect 130 predictions; pilot expects 6,500; main expects 26,000. Verify all 13 conditions complete and the original K=32 configuration is restored before proceeding.
- Run `score_results.ipynb` with the matching `STAGE` and the exact `EXPERIMENT_DIR` printed above. The scorer currently defaults to `main`; set it to `smoke` or `pilot` for those runs. The existing prediction and manifest schema is retained.
- Keep NVIDIA and AMD run directories separate, and report accuracy and efficiency by hardware. Never overwrite or combine the raw timing measurements from the two systems.
- The optional leave-one-query-out experiment will use a separate notebook and results root. It will compare each K=31 condition with an unmodified K=32 baseline on the same evaluation questions. The main scoring notebook does not depend on that experiment.